# Types of Machine Learning

Machine learning is not a single technique — it is a family of approaches united by the idea of learning from data. What distinguishes one approach from another is the **type of signal** available during training: whether correct answers are provided, whether structure must be discovered without guidance, or whether feedback comes in the form of rewards and penalties from an environment.

Understanding these types is essential because the choice of learning paradigm determines which algorithms you can use, what data you need to collect, how you evaluate success, and what kinds of problems you can solve. A spam filter requires labeled examples of spam and legitimate email — that is supervised learning. Customer segmentation from purchase history without predefined groups — that is unsupervised learning. A game-playing agent that improves through trial and error — that is reinforcement learning.

This notebook examines each major type of machine learning in depth: what defines it, how it works, what problems it solves, which algorithms belong to it, and how the types relate to one another.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification, make_blobs, make_regression
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, mean_squared_error, silhouette_score

np.set_printoptions(precision=4, suppress=True)
plt.style.use("seaborn-v0_8-whitegrid")
np.random.seed(42)

---

## 1. The Machine Learning Taxonomy

Machine learning algorithms are classified primarily by **what the training data provides**. The following diagram shows the main branches:

```
                    MACHINE LEARNING
                          │
        ┌─────────────────┼─────────────────┐
        │                 │                 │
   SUPERVISED        UNSUPERVISED      REINFORCEMENT
   (has labels)      (no labels)       (reward signal)
        │                 │
   ┌────┴────┐      ┌─────┴─────┐
Regression  Classification  Clustering  Dim. Reduction
        │
   SEMI-SUPERVISED (few labels + many unlabeled)
   SELF-SUPERVISED (labels derived from data itself)
```

The boundaries are not always rigid. Semi-supervised learning blends supervised and unsupervised ideas. Self-supervised learning is technically unsupervised but produces representations useful for supervised downstream tasks. Transfer learning applies knowledge from one task to another. In practice, you choose the paradigm that matches your data and problem, then select specific algorithms within that paradigm.

---

## 2. Supervised Learning

Supervised learning is the most widely used paradigm in machine learning. The algorithm learns from **labeled training data** — examples where both the input and the correct output are known. The model's job is to learn the mapping from inputs to outputs so it can predict the correct output for new, unseen inputs.

Formally, given a training set of $n$ examples $\{(x_1, y_1), (x_2, y_2), \ldots, (x_n, y_n)\}$, supervised learning finds a function $f$ such that $f(x_i) \approx y_i$ for training examples and generalizes to new $x$ values.

The "supervision" comes from the labels — they act as a teacher telling the model the right answer for each example. The model compares its predictions to the labels, computes error, and adjusts to reduce that error.

### 2.1 Regression

**Regression** predicts a **continuous numerical value**. The output can be any real number: a house price of \$284,500, a temperature of 22.7°C, a probability of 0.73.

Common regression algorithms:
- **Linear regression** — fits a straight line (or hyperplane) to data.
- **Polynomial regression** — fits curves by adding polynomial features.
- **Ridge / Lasso regression** — linear regression with regularization to prevent overfitting.
- **Decision tree regression** — piecewise constant predictions via tree splits.
- **Random forest regression** — ensemble of regression trees.

Evaluation metrics: Mean Squared Error (MSE), Root Mean Squared Error (RMSE), Mean Absolute Error (MAE), R² (coefficient of determination).

### 2.2 Classification

**Classification** predicts a **discrete category or class label**. The output is one of a finite set: spam or ham, cat or dog or bird, approved or denied.

- **Binary classification** — two classes (yes/no, true/false, positive/negative).
- **Multi-class classification** — three or more classes (digit 0–9, species of flower).
- **Multi-label classification** — multiple labels per example (a document tagged with several topics).

Common classification algorithms:
- **Logistic regression** — despite the name, used for classification; outputs probabilities.
- **K-Nearest Neighbors (KNN)** — classifies by majority vote of nearest training examples.
- **Decision trees** — sequential splits on features.
- **Support Vector Machines (SVM)** — finds optimal boundary between classes.
- **Naive Bayes** — probabilistic classifier based on Bayes' theorem.
- **Random Forest / Gradient Boosting** — ensemble methods.

Evaluation metrics: Accuracy, precision, recall, F1 score, ROC-AUC, confusion matrix.

In [ ]:
# Supervised Learning — Regression
X_reg, y_reg = make_regression(n_samples=150, n_features=1, noise=15, random_state=42)

reg = LinearRegression().fit(X_reg, y_reg)
X_test_reg = np.linspace(X_reg.min(), X_reg.max(), 100).reshape(-1, 1)

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.scatter(X_reg, y_reg, alpha=0.5, s=20)
plt.plot(X_test_reg, reg.predict(X_test_reg), "r-", linewidth=2)
plt.xlabel("Feature")
plt.ylabel("Target (continuous)")
plt.title("Regression: predict a number")

# Supervised Learning — Classification
X_cls, y_cls = make_classification(n_samples=200, n_features=2, n_redundant=0,
                                    n_clusters_per_class=1, random_state=42)
clf = LogisticRegression().fit(X_cls, y_cls)

plt.subplot(1, 2, 2)
plt.scatter(X_cls[:, 0], X_cls[:, 1], c=y_cls, cmap="coolwarm", alpha=0.6, edgecolors="k", linewidth=0.3)
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("Classification: predict a category")

plt.tight_layout()
plt.show()
print(f"Regression R² on training data: {reg.score(X_reg, y_reg):.4f}")
print(f"Classification accuracy: {clf.score(X_cls, y_cls):.4f}")

### 2.3 Characteristics and Requirements of Supervised Learning

Supervised learning requires **labeled data**, which is often the bottleneck. Labeling is expensive and time-consuming: a radiologist must mark tumors on X-rays, a linguist must tag parts of speech, a user must mark emails as spam. The quality of labels directly affects model quality — mislabeled training examples teach the model wrong patterns.

The amount of labeled data needed depends on problem complexity. Simple problems (linearly separable classes) may need hundreds of examples. Complex problems (image recognition with thousands of categories) may need millions. Techniques like data augmentation, transfer learning, and semi-supervised learning address label scarcity.

Supervised learning dominates industry because the objective is clear: minimize prediction error against known correct answers. This clarity makes evaluation straightforward and makes supervised learning applicable to a vast range of business problems — any situation where historical outcomes are recorded and future outcomes need to be predicted.

---

## 3. Unsupervised Learning

Unsupervised learning works with **unlabeled data**. The algorithm receives inputs without correct answers and must discover **hidden structure** on its own. There is no teacher — the model finds patterns, groupings, or representations without external guidance about what the "right" output should be.

Unsupervised learning answers questions like: *Are there natural groups in this data? Can we compress this data without losing important information? Which data points are unusual?*

### 3.1 Clustering

**Clustering** groups similar data points together. Points within a cluster are more similar to each other than to points in other clusters. The algorithm does not know group names in advance — it discovers how many groups exist and which points belong where.

Applications: customer segmentation for marketing, document grouping by topic, image segmentation, gene sequence analysis, anomaly grouping.

Algorithms: **K-Means** (partition data into K clusters), **Hierarchical clustering** (build a tree of clusters), **DBSCAN** (density-based, finds clusters of arbitrary shape).

### 3.2 Dimensionality Reduction

**Dimensionality reduction** reduces the number of features while preserving important information. High-dimensional data is hard to visualize, expensive to compute on, and prone to overfitting. Reduction techniques compress data to fewer dimensions.

Applications: visualizing high-dimensional data in 2D or 3D, noise removal, feature extraction before supervised learning, data compression.

Algorithms: **PCA** (Principal Component Analysis — linear projection to directions of maximum variance), **t-SNE** (nonlinear visualization), **Autoencoders** (neural network compression).

### 3.3 Anomaly Detection

**Anomaly detection** identifies data points that deviate significantly from the norm. Unlike clustering, the focus is on the rare unusual points rather than the group structure.

Applications: fraud detection, network intrusion detection, manufacturing defect detection, medical abnormality flagging.

Algorithms: **Isolation Forest**, **One-Class SVM**, statistical methods (Z-score thresholding).

In [ ]:
# Unsupervised Learning — Clustering (K-Means)
X_cluster, _ = make_blobs(n_samples=300, centers=4, cluster_std=0.8, random_state=42)

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
labels = kmeans.fit_predict(X_cluster)

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.scatter(X_cluster[:, 0], X_cluster[:, 1], c=labels, cmap="viridis", alpha=0.7, edgecolors="k", linewidth=0.3)
plt.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1],
            c="red", marker="X", s=150, linewidths=2, label="Centroids")
plt.title(f"K-Means Clustering (K=4, silhouette={silhouette_score(X_cluster, labels):.3f})")
plt.legend()

# Unsupervised Learning — Dimensionality Reduction (PCA)
X_high, _ = make_classification(n_samples=200, n_features=20, n_informative=5, random_state=42)
pca = PCA(n_components=2)
X_2d = pca.fit_transform(X_high)

plt.subplot(1, 2, 2)
plt.scatter(X_2d[:, 0], X_2d[:, 1], alpha=0.6)
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)")
plt.title("PCA: 20 features → 2 dimensions")

plt.tight_layout()
plt.show()

### 3.4 Evaluating Unsupervised Learning

Evaluating unsupervised learning is harder than supervised learning because there are no labels to compare against. Metrics are indirect:

- **Silhouette score** — measures how similar a point is to its own cluster versus other clusters. Range $[-1, 1]$; higher is better.
- **Inertia** (K-Means) — sum of squared distances to centroids. Lower is better, but always decreases with more clusters.
- **Explained variance** (PCA) — proportion of total variance captured by reduced dimensions.
- **Downstream task performance** — use unsupervised output as input to a supervised model and measure if it improves.

Despite evaluation challenges, unsupervised learning is valuable when labels are unavailable, expensive, or unknown — and for exploratory analysis that reveals structure humans did not anticipate.

---

## 4. Reinforcement Learning

Reinforcement learning (RL) trains an **agent** to make **sequential decisions** in an **environment** by maximizing cumulative **reward**. Unlike supervised learning, there are no labeled input-output pairs. Unlike unsupervised learning, the goal is not to find structure but to learn an optimal **policy** — a strategy for choosing actions.

The agent interacts with the environment in a loop:

```
  Agent → Action → Environment → New State + Reward → Agent → ...
```

At each step, the agent observes the current **state**, chooses an **action**, receives a **reward** (positive for good outcomes, negative for bad), and transitions to a new state. The goal is to maximize total reward over time — not just immediate reward but long-term cumulative reward.

### 4.1 Key Concepts

**State** — A description of the current situation (chess board position, robot location, game screen pixels).

**Action** — What the agent can do (move a chess piece, steer left, press a button).

**Reward** — Numerical feedback signal (+1 for winning, -1 for losing, +0.01 for staying alive).

**Policy** — The agent's strategy: a mapping from states to actions. $\pi(s) \rightarrow a$.

**Value function** — Expected cumulative reward from a state. Helps the agent plan beyond immediate rewards.

**Exploration vs exploitation** — The agent must balance trying new actions (exploration) with using known good actions (exploitation). This tradeoff is fundamental to RL.

### 4.2 Applications

- **Game playing** — Chess, Go, Atari, StarCraft (AlphaGo, OpenAI Five).
- **Robotics** — Walking, grasping, navigation through trial and error.
- **Recommendation systems** — Learning which items to suggest to maximize engagement.
- **Resource management** — Data center cooling, traffic light timing, portfolio management.
- **RLHF** — Reinforcement Learning from Human Feedback, used to align large language models with human preferences.

### 4.3 RL Algorithms (Overview)

- **Q-Learning** — Learns the value of state-action pairs in a table or function approximator.
- **Deep Q-Networks (DQN)** — Q-learning with neural networks for complex state spaces.
- **Policy Gradient methods** — Directly optimize the policy.
- **Actor-Critic** — Combines value estimation with policy optimization.
- **PPO, A3C** — Modern policy gradient variants used in robotics and games.

In [ ]:
# Reinforcement Learning — simplified multi-armed bandit (explore vs exploit)
# 3 slot machines with different true win rates; agent learns which is best

true_rates = np.array([0.3, 0.7, 0.5])  # machine 1 is best
n_machines = len(true_rates)
n_rounds = 500
epsilon = 0.1  # explore 10% of the time

counts = np.zeros(n_machines)
values = np.zeros(n_machines)  # estimated win rates
rewards_history = []

for t in range(n_rounds):
    if np.random.random() < epsilon:
        action = np.random.randint(n_machines)  # explore
    else:
        action = np.argmax(values)  # exploit

    reward = 1 if np.random.random() < true_rates[action] else 0
    counts[action] += 1
    values[action] += (reward - values[action]) / counts[action]  # incremental mean
    rewards_history.append(reward)

cumulative_reward = np.cumsum(rewards_history)
running_avg = cumulative_reward / np.arange(1, n_rounds + 1)

plt.figure(figsize=(10, 4))
plt.plot(running_avg, linewidth=1.5)
plt.axhline(true_rates.max(), color="r", linestyle="--", label=f"Best machine rate ({true_rates.max()})")
plt.xlabel("Round")
plt.ylabel("Average reward")
plt.title("RL: Epsilon-Greedy Bandit learns best machine (0.7 win rate)")
plt.legend()
plt.show()

print("Estimated win rates:", np.round(values, 3))
print("True win rates:     ", true_rates)
print(f"Most played machine: {np.argmax(counts)} (best = 1)")

---

## 5. Semi-Supervised Learning

Semi-supervised learning combines a **small amount of labeled data** with a **large amount of unlabeled data**. Labeling is expensive — a medical expert labeling X-rays costs time and money. But unlabeled data (X-rays without diagnoses) may be abundant.

The intuition is that unlabeled data reveals the **structure of the input distribution** — which regions of feature space are dense, which are sparse, how clusters form. This structural information helps the model generalize better than using only the small labeled set.

**Assumptions** that make semi-supervised learning work:

- **Smoothness assumption** — Points close together likely have the same label.
- **Cluster assumption** — Data forms clusters, and points in the same cluster share a label.
- **Manifold assumption** — High-dimensional data lies on a lower-dimensional structure.

**Applications:** Web page classification (few labeled, many unlabeled), protein sequence classification, speech recognition with limited transcripts.

**Techniques:** Self-training (model labels unlabeled data iteratively), label propagation (spread labels through a graph), pseudo-labeling (use model predictions as training labels for unlabeled data).

In [ ]:
# Semi-supervised: train on few labels vs many labels
X, y = make_classification(n_samples=500, n_features=2, n_redundant=0, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Only 20 labeled samples; rest treated as unlabeled (simulated by training on subset)
n_labeled = 20
X_labeled = X_train[:n_labeled]
y_labeled = y_train[:n_labeled]

clf_few = KNeighborsClassifier(n_neighbors=3).fit(X_labeled, y_labeled)
clf_many = KNeighborsClassifier(n_neighbors=5).fit(X_train, y_train)

acc_few = accuracy_score(y_test, clf_few.predict(X_test))
acc_many = accuracy_score(y_test, clf_many.predict(X_test))

print(f"KNN with {n_labeled} labeled samples:  test accuracy = {acc_few:.3f}")
print(f"KNN with {len(X_train)} labeled samples: test accuracy = {acc_many:.3f}")
print("\nSemi-supervised methods use unlabeled data to improve beyond the few-label baseline.")

---

## 6. Self-Supervised Learning

Self-supervised learning creates **labels from the data itself**, eliminating the need for human annotation. It is a form of unsupervised learning that produces representations useful for downstream supervised tasks.

The core idea: design a **pretext task** where the input contains both the question and the answer. The model learns useful features by solving these artificial tasks, then transfers those features to real tasks.

**Examples of pretext tasks:**

- **Next word prediction** — Given "The cat sat on the," predict "mat." This is how large language models are pre-trained. The text itself provides the labels.
- **Masked language modeling** — Hide random words and predict them (BERT).
- **Image inpainting** — Hide part of an image and predict the missing pixels.
- **Contrastive learning** — Learn that two augmented versions of the same image should have similar representations (SimCLR, MoCo).

Self-supervised learning has driven the biggest advances in modern AI. GPT, BERT, and vision transformers all rely on self-supervised pre-training on massive unlabeled datasets, followed by fine-tuning on smaller labeled datasets for specific tasks. This paradigm makes it possible to leverage the enormous quantities of unlabeled text and images on the internet.

In [ ]:
# Self-supervised concept: next-word prediction from raw text
corpus = "the cat sat on the mat the dog sat on the log"
words = corpus.split()

# Build (context, next_word) pairs — labels come from the text itself
pairs = [(" ".join(words[i:i+2]), words[i+2]) for i in range(len(words) - 2)]

print("Self-supervised training pairs (context → target):")
for context, target in pairs[:6]:
    print(f"  '{context}' → '{target}'")
print(f"  ... ({len(pairs)} total pairs from one sentence)")
print("\nScale this to billions of sentences → pre-train a language model.")

---

## 7. Comparing the Types

| Aspect | Supervised | Unsupervised | Reinforcement |
|--------|------------|--------------|---------------|
| **Training signal** | Labels (correct answers) | No labels | Rewards / penalties |
| **Goal** | Predict output for new inputs | Discover structure | Maximize cumulative reward |
| **Data needed** | Labeled examples | Unlabeled examples | Environment to interact with |
| **Output** | Prediction (number or class) | Clusters, reduced dimensions | Action policy |
| **Evaluation** | Accuracy, MSE, F1, etc. | Silhouette, variance, downstream | Total reward, win rate |
| **Examples** | Spam filter, price prediction | Customer segments, PCA | Game AI, robotics |
| **Label cost** | High | None | None (but environment design) |

| Aspect | Semi-Supervised | Self-Supervised |
|--------|-----------------|-----------------|
| **Training signal** | Few labels + many unlabeled | Labels derived from data |
| **Goal** | Better supervised model with less labeling | Learn representations, then fine-tune |
| **Examples** | Web classification, medical imaging | GPT, BERT, SimCLR |

---

## 8. Choosing the Right Type

The choice of learning type follows directly from what data and feedback you have.

**Choose supervised learning when:**
- You have labeled historical data with known outcomes.
- The goal is to predict a specific output (price, category, probability).
- Examples: predicting sales, classifying emails, diagnosing diseases from labeled scans.

**Choose unsupervised learning when:**
- You have data but no labels, and want to explore structure.
- You need to segment, compress, or detect anomalies.
- Examples: customer segmentation, data visualization, fraud anomaly detection.

**Choose reinforcement learning when:**
- The problem involves sequential decisions over time.
- Feedback comes as rewards, not labels.
- An environment can be simulated or accessed for trial and error.
- Examples: game playing, robot control, dynamic pricing, ad placement.

**Choose semi-supervised learning when:**
- Labels are scarce but unlabeled data is abundant.
- Labeling cost is prohibitive.
- Examples: medical imaging with few expert annotations, web content classification.

**Choose self-supervised learning when:**
- Massive unlabeled data exists (text, images).
- You want to pre-train representations for multiple downstream tasks.
- Examples: language model pre-training, image representation learning.

---

## 9. How the Types Connect in Practice

Real-world AI systems often combine multiple learning types in a pipeline.

A **recommendation system** might use unsupervised clustering to segment users, supervised learning to predict click probability, and reinforcement learning to optimize long-term engagement versus short-term clicks.

A **language model** is pre-trained with self-supervised learning on billions of text documents, then fine-tuned with supervised learning on labeled task-specific data, then further aligned with reinforcement learning from human feedback (RLHF).

A **medical imaging system** might use unsupervised PCA to reduce dimensionality, semi-supervised learning when only some scans are expert-labeled, and supervised classification for the final diagnosis.

Understanding the types is not about picking exactly one — it is about recognizing which learning signal is available at each stage and choosing the appropriate paradigm for each sub-problem.

In [ ]:
# Side-by-side: same dataset, three paradigms
X, y = make_blobs(n_samples=300, centers=3, random_state=42)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Supervised: use labels to classify
clf = KNeighborsClassifier(n_neighbors=5).fit(X, y)
axes[0].scatter(X[:, 0], X[:, 1], c=clf.predict(X), cmap="viridis", alpha=0.7)
axes[0].set_title("Supervised\n(uses labels to classify)")

# Unsupervised: discover clusters without labels
km = KMeans(n_clusters=3, random_state=42, n_init=10).fit(X)
axes[1].scatter(X[:, 0], X[:, 1], c=km.labels_, cmap="viridis", alpha=0.7)
axes[1].set_title("Unsupervised\n(discovers 3 clusters)")

# Ground truth labels (what supervised had access to)
axes[2].scatter(X[:, 0], X[:, 1], c=y, cmap="viridis", alpha=0.7)
axes[2].set_title("True labels\n(ground truth)")

for ax in axes:
    ax.set_xlabel("Feature 1")
    ax.set_ylabel("Feature 2")

plt.tight_layout()
plt.show()
print(f"Supervised accuracy vs true labels: {accuracy_score(y, clf.predict(X)):.3f}")
print(f"Unsupervised cluster purity (approx): labels match well when K=3")

---

## 10. Summary

Machine learning is organized into types based on the training signal available. **Supervised learning** uses labeled data to predict outputs — regression for continuous values, classification for categories. **Unsupervised learning** finds structure in unlabeled data through clustering, dimensionality reduction, and anomaly detection. **Reinforcement learning** trains agents to maximize reward through interaction with an environment.

**Semi-supervised learning** leverages both few labels and abundant unlabeled data. **Self-supervised learning** derives labels from the data itself, powering modern language and vision model pre-training.

The choice of type depends on what data you have, what feedback is available, and what problem you are solving. In practice, sophisticated systems combine multiple types across different stages of the pipeline — pre-training representations, fine-tuning on labels, and optimizing behavior through rewards — each stage using the paradigm best suited to the signal available at that point.